# Zoo Exam Code-Only Notebook
Student: Manjunath S

In [ ]:
import pandas as pd
import numpy as np
import json, re, os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
ROLL = "24UG00319"
SEAT = 18

method_prefix = f"{SEAT}_"
last_digit = int(ROLL[-1])
second_last = int(ROLL[-2])
third_last = int(ROLL[-3])

mod3 = sum(int(c) for c in ROLL if c.isdigit()) % 3
mod5 = sum(int(c) for c in ROLL if c.isdigit()) % 5

print(method_prefix, last_digit, second_last, third_last, mod3, mod5)

In [ ]:
class ZooPipeline:
    def clean_name(self, name):
        import re
        name = str(name).replace(" ","")
        return re.sub(r'[^A-Za-z0-9]',"",name)

    def habitat_risk(self, h):
        h=str(h).lower()
        if "marine" in h: return 3
        if "fresh" in h: return 2
        return 1

    def trophic_level(self, d):
        d=str(d).lower()
        if d=="carnivore": return 3
        if d=="omnivore": return 2
        return 1

In [ ]:
pipeline = ZooPipeline()

zoo = pd.read_csv("extracted/zoo.csv")
zoo["animal_name"] = zoo["animal_name"].apply(pipeline.clean_name)

cls = pd.read_csv("extracted/class.csv")

with open("extracted/auxiliary_metadata_cleaned.json") as f:
    aux = json.load(f)
aux_df = pd.DataFrame(aux)
aux_df["animal_name"] = aux_df["animal_name"].apply(pipeline.clean_name)

merged = zoo.merge(cls, left_on="class_type", right_on="Class_Number", how="left")
merged_final = merged.merge(aux_df, on="animal_name", how="left")

merged_final.head()

In [ ]:
categorical_cols = merged_final.select_dtypes(include="object").columns
numeric_cols = merged_final.select_dtypes(include=["int64","float64"]).columns

merged_final[categorical_cols] = merged_final[categorical_cols].fillna("unknown")
for col in numeric_cols:
    merged_final[col] = merged_final[col].fillna(merged_final[col].median())

merged_final["habitat_risk_score"] = merged_final["habitat_type"].apply(pipeline.habitat_risk)
merged_final["trophic_level"] = merged_final["diet"].apply(pipeline.trophic_level)

merged_final.head()

In [ ]:
X = merged_final.select_dtypes(include=["int64","float64"]).drop(columns=["class_type","Class_Number"])
y = merged_final["class_type"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=123)

rf = RandomForestClassifier(n_estimators=80, max_depth=8, min_samples_split=6, random_state=123)
rf.fit(X_train, y_train)

print("RF Accuracy:", accuracy_score(y_test, rf.predict(X_test)))

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

print("KNN Accuracy:", accuracy_score(y_test, knn.predict(X_test)))